# 🚀 Google Colab LLM Offload for Banking Assistant

This notebook will:
1. Install and configure Ollama in Colab
2. Mount Google Drive to access your GGUF model
3. Create dynamic Modelfile and register your model
4. Start Ollama server with ngrok exposure
5. Generate public ngrok URL for your local FastAPI

**Your local FastAPI backend remains unchanged** — only the `OLLAMA_BASE_URL` environment variable will point to Colab!

---
### 📋 Prerequisites
- Your GGUF model file should be uploaded to Google Drive
- Local FastAPI continues running normally
- No changes needed to your existing backend code

## 🔧 Step 1: Install Dependencies and Mount Drive

In [1]:
# Install required packages
!pip install -q pyngrok

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive mounted successfully!')
print('📁 Drive path: /content/drive/MyDrive')

Mounted at /content/drive
✅ Google Drive mounted successfully!
📁 Drive path: /content/drive/MyDrive


## 📦 Step 2: Setup Model Path and Variables

In [2]:
import os
import subprocess
import time
import requests

# ✏️ CHANGE THIS to your actual model filename
MODEL_FILENAME = "gemma-2b-banking-q4_k_m.gguf"

DRIVE_PATH = "/content/drive/MyDrive"
COLAB_MODEL_PATH = f"{DRIVE_PATH}/{MODEL_FILENAME}"
OLLAMA_PORT = 11434

# Check if model exists in Google Drive
if not os.path.exists(COLAB_MODEL_PATH):
    print(f"❌ Model file not found: {MODEL_FILENAME}")
    print(f"📁 Please upload {MODEL_FILENAME} to your Google Drive root folder")
    print(f"📁 Expected path: {COLAB_MODEL_PATH}")
else:
    model_size_gb = os.path.getsize(COLAB_MODEL_PATH) / (1024**3)
    print(f"✅ Model found: {MODEL_FILENAME}")
    print(f"📁 Path: {COLAB_MODEL_PATH}")
    print(f"📦 Size: {model_size_gb:.2f} GB")

✅ Model found: gemma-2b-banking-q4_k_m.gguf
📁 Path: /content/drive/MyDrive/gemma-2b-banking-q4_k_m.gguf
📦 Size: 1.52 GB


## 🤖 Step 3: Install Ollama

In [3]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama installed!')

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
✅ Ollama installed!


In [4]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed correctly!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,056 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently

## 🚀 Step 4: Start Ollama Server and Load Model

In [5]:
import subprocess
import time
import os

# Kill any existing Ollama processes
os.system('pkill -f ollama || true')
time.sleep(2)

# Start Ollama server in background
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

print('⏳ Waiting for Ollama server to start...')
time.sleep(8)

# Check if it started
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    print('✅ Ollama server is running!')
except:
    print('❌ Ollama server did not start. Try re-running this cell.')

⏳ Waiting for Ollama server to start...
✅ Ollama server is running!


In [15]:
modelfile_content = f"""FROM {COLAB_MODEL_PATH}

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER num_predict 150

PARAMETER stop "### Instruction:"
PARAMETER stop "<|end|>"
PARAMETER stop "</s>"

SYSTEM \"\"\"You are a professional banking and sales assistant. You provide accurate,
helpful responses about banking products, account inquiries, loan applications,
and sales-related questions. You were fine-tuned on real banking and sales
conversation data to understand customer needs and provide appropriate solutions.
Keep your responses focussed.\"\"\"
"""

with open('/content/Modelfile', 'w') as f:
    f.write(modelfile_content)

print('📄 Modelfile created. Re-registering model...')

result = subprocess.run(
    ['ollama', 'create', 'banking-model', '-f', '/content/Modelfile'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ Model re-registered successfully!')
else:
    print('❌ Error:')
    print(result.stderr)

📄 Modelfile created. Re-registering model...
✅ Model re-registered successfully!


## 🌐 Step 5: Setup Ngrok Tunnel

In [10]:
from pyngrok import ngrok
import time

# Paste your token here
NGROK_AUTH_TOKEN = "3BAvo7pdnsT4QOxCgaaQ3RVUt0Q_7fKngxaiUmR6ExxqTock7"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
time.sleep(2)

try:
    tunnel = ngrok.connect(11434)
    ngrok_url = tunnel.public_url
    print(f'✅ Ngrok tunnel created!')
    print(f'🔗 Public URL: {ngrok_url}')
except Exception as e:
    print(f'❌ Ngrok error: {e}')
    ngrok_url = None

✅ Ngrok tunnel created!
🔗 Public URL: https://unexercisable-wabbly-gala.ngrok-free.dev


## 📋 Step 6: Get Your Environment Variables

In [11]:
if ngrok_url:
    session_id = f'colab_session_{int(time.time())}'

    print('=' * 60)
    print('🔧 RUN THESE COMMANDS IN YOUR LOCAL TERMINAL:')
    print('=' * 60)
    print(f"export OLLAMA_BASE_URL='{ngrok_url}'")
    print(f"export COLAB_SESSION_ID='{session_id}'")
    print()
    print('Then restart your FastAPI:')
    print('uvicorn main:app --host 0.0.0.0 --port 8002 --reload')
    print('=' * 60)

    # Save to file
    with open('/content/env_vars.txt', 'w') as f:
        f.write(f"export OLLAMA_BASE_URL='{ngrok_url}'\n")
        f.write(f"export COLAB_SESSION_ID='{session_id}'\n")

    print('💾 Also saved to /content/env_vars.txt')
else:
    print('❌ ngrok URL not available. Re-run Step 5.')

🔧 RUN THESE COMMANDS IN YOUR LOCAL TERMINAL:
export OLLAMA_BASE_URL='https://unexercisable-wabbly-gala.ngrok-free.dev'
export COLAB_SESSION_ID='colab_session_1773945119'

Then restart your FastAPI:
uvicorn main:app --host 0.0.0.0 --port 8002 --reload
💾 Also saved to /content/env_vars.txt


## 🔍 Step 7: Health Check

In [12]:
import requests

print('=' * 50)
print('🔍 HEALTH CHECKS')
print('=' * 50)

# Check Ollama locally
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = r.json().get('models', [])
    print(f'✅ Ollama local: OK ({len(models)} model(s) loaded)')
    for m in models:
        print(f'   - {m["name"]}')
except Exception as e:
    print(f'❌ Ollama local: {e}')

# Check via ngrok
if ngrok_url:
    try:
        r = requests.get(f'{ngrok_url}/api/tags', timeout=10)
        print(f'✅ Ngrok tunnel: OK')
        print(f'🔗 Public URL: {ngrok_url}')
    except Exception as e:
        print(f'❌ Ngrok tunnel: {e}')

print('=' * 50)
print('🟢 Setup complete! Your local backend can now use Colab LLM.')

🔍 HEALTH CHECKS
✅ Ollama local: OK (1 model(s) loaded)
   - banking-model:latest
✅ Ngrok tunnel: OK
🔗 Public URL: https://unexercisable-wabbly-gala.ngrok-free.dev
🟢 Setup complete! Your local backend can now use Colab LLM.


## 📥 Step 8: Download env_vars.txt

In [ ]:
from google.colab import files

try:
    files.download('/content/env_vars.txt')
    print('✅ Download started!')
except Exception as e:
    print(f'❌ Download error: {e}')
    print('You can manually download from the Files panel on the left.')

In [14]:
import requests

response = requests.post('http://localhost:11434/api/generate',
    json={
        "model": "banking-model",
        "prompt": "### Instruction:\nWhat are your savings account interest rates?\n\n### Response:\n",
        "stream": False
    },
    timeout=120
)

print(repr(response.json()['response']))

"I'd be happy to help you with that! Here's the information on our interest rates for savings accounts:\n\nAccount Type | Annual Percentage Rate (APR)\nBasic Savings | 0.00%\nInterest Checking | 0.00%\nPremium Savings | 0.00%\nYouth Savings | 1.50%\nSenior Savings | 2.00%\nCollege Savings | 1.00%\n\nThese rates may vary based on the type of account and any special promotions we may be offering. If you have specific questions or need more information, feel free to let me know!"


## ⚠️ Keep This Tab Open!

- Colab session stays alive as long as this tab is open
- If ngrok URL changes (session restart), re-run Steps 5 & 6
- Free Colab sessions last up to **12 hours**
- Free ngrok has connection limits — get a free auth token at https://ngrok.com for better stability